# Analyzing Correlation between MODIS AOD and AQI

This notebook merges ground-based hourly AQI measurements with daily MCD19A2 MODIS AOD data. 

Because MODIS passes over at specific times (e.g., 10:15, 13:30) and AQI is recorded strictly hourly (e.g., 10:00, 13:00), we use `pandas.merge_asof` to match each satellite reading to the nearest ground station reading.

In [1]:
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import pyproj
import rasterio
from pyhdf.SD import SD, SDC

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Prepare AQI Data

In [ ]:
station_df = pd.read_csv('/home/work1/projects/Air_Quality/Masterdata/target_stations_aod.csv')

In [4]:
station_df

,Name,ID,Latitude,Longitude
0,"Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...",31388839920718814259329251882,10.99230,106.65770
1,Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...,31387251434693138681789561386,21.30150,106.22603
2,HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trư...,31390916083317566102523755051,10.78230,106.68340
3,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,31390912357075263208060500522,10.78230,106.75280
4,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,31388883344354363840031242796,20.53600,105.91650
5,Hà Nội: 556 Nguyễn Văn Cừ (KK),28560877461938780203765592307,21.04910,105.88310
6,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,31390908889087377344742439468,21.00310,105.79470
7,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),31390903576425084107499649578,21.00520,105.84180
8,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,31390932574706768021562473002,10.53910,106.40450
9,Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK),28505268571336961948594948504,21.33847,105.36330


In [5]:
# 2. Setup Coordinate Transformation (Lat/Lon EPSG:4326 to MODIS Sinusoidal)
modis_crs = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"
transformer = pyproj.Transformer.from_crs("EPSG:4326", modis_crs, always_xy=True)

# Calculate MODIS X and Y coordinates
station_df['modis_x'], station_df['modis_y'] = transformer.transform(
    station_df['Longitude'].values, 
    station_df['Latitude'].values
    )

station_df

,Name,ID,Latitude,Longitude,modis_x,modis_y
0,"Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...",31388839920718814259329251882,10.99230,106.65770,1.164221e+07,1.222289e+06
1,Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...,31387251434693138681789561386,21.30150,106.22603,1.100485e+07,2.368621e+06
2,HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trư...,31390916083317566102523755051,10.78230,106.68340,1.165323e+07,1.198938e+06
3,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,31390912357075263208060500522,10.78230,106.75280,1.166081e+07,1.198938e+06
4,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,31388883344354363840031242796,20.53600,105.91650,1.102896e+07,2.283502e+06
5,Hà Nội: 556 Nguyễn Văn Cừ (KK),28560877461938780203765592307,21.04910,105.88310,1.098805e+07,2.340556e+06
6,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,31390908889087377344742439468,21.00310,105.79470,1.098227e+07,2.335441e+06
7,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),31390903576425084107499649578,21.00520,105.84180,1.098700e+07,2.335674e+06
8,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,31390932574706768021562473002,10.53910,106.40450,1.163206e+07,1.171896e+06
9,Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK),28505268571336961948594948504,21.33847,105.36330,1.091272e+07,2.372732e+06


In [ ]:
path = "/home/slow_data/Air_Quality/station_historical_full"

# Load AQI
df_aqi = pd.read_csv(path)

# Convert timestamp
df_aqi['Timestamp_DT'] = pd.to_datetime(df_aqi['Timestamp'], format='%d/%m/%Y %H:%M')

# Ensure ID is string to prevent merge conflicts
df_aqi['ID'] = df_aqi['ID'].astype(str)

# IMPORTANT for merge_asof: Data must be sorted by time
df_aqi = df_aqi.sort_values(by='Timestamp_DT').reset_index(drop=True)

df_aqi.head()

,Source,ID,Timestamp,Name,Latitude,Longitude,AQI,PM2.5,PM10,CO,NO2,O3,SO2,Temperature,Humidity,Pressure,Wind Speed,Timestamp_DT
0,gov,28560877461938780203765592307,08/04/2025 14:00,Hà Nội: 556 Nguyễn Văn Cừ (KK),21.0491,105.8831,134,134.333871,80.967371,8.601812,52.59890,8.319781,5.55468,27.96,71.0,1012.0,3.90,2025-04-08 14:00:00
1,gov,31390912357075263208060500522,08/04/2025 14:00,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,10.7823,106.7528,95,95.340935,59.552588,NaN,8.45665,NaN,16.50532,35.01,46.0,1009.0,1.54,2025-04-08 14:00:00
2,gov,31390932574706768021562473002,08/04/2025 14:00,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,10.5391,106.4045,89,88.681186,52.501889,29.512184,NaN,NaN,1.95932,29.45,53.0,1009.0,5.84,2025-04-08 14:00:00
3,gov,31390903576425084107499649578,08/04/2025 14:00,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),21.0052,105.8418,151,151.418019,72.311020,NaN,12.98290,9.472219,2.94200,28.00,70.0,1012.0,3.85,2025-04-08 14:00:00
4,gov,31390908889087377344742439468,08/04/2025 14:00,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,21.0031,105.7947,107,106.968661,71.106262,12.374913,4.44125,9.902344,2.29400,28.01,69.0,1012.0,3.68,2025-04-08 14:00:00


## 2. Helper Logic: Extracting Exact Time from MODIS MCD19A2
MODIS stores orbit times as a string of UTC times (e.g. `"0315 0455"`) in the `Orbit_time_stamp` attribute. We need to parse this and convert it to local time.

In [6]:

# # 1. Parse Date from filename (A + Year + DayOfYear)
# match = re.search(r'A(\d{4})(\d{3})', filename)
# if not match:
#     raise ValueError("Filename does not match expected MODIS format.")

# year, doy = match.groups()
# base_date = datetime(int(year), 1, 1) + timedelta(days=int(doy) - 1)
    


In [7]:
def parse_modis_orbit_times(orbit_time_str, tz_offset_hours=7):
    """
    Extracts actual local datetimes for each orbit layer in a MODIS HDF file.
    
    Args:
        orbit_time_str (str): Value from hdf.attributes()['Orbit_time_stamp'] 
        Orbit_time_stamp format:YYYY (Year), DDD (Day of Year), HH (Hour), MM (Minute), T/A (Satellite)

        tz_offset_hours (int): Timezone offset (7 for Vietnam UTC+7)
        
    Returns:
        list: List of local datetimes corresponding to each orbit layer.
    """
    if isinstance(orbit_time_str, bytes):
        orbit_time_str = orbit_time_str.decode('utf-8')
        
    orbit_times = str(orbit_time_str).strip().split()
    local_datetimes = []
    
    for t in orbit_times:
        if len(t) >= 11:
            year = int(t[0:4])
            doy = int(t[4:7])     # Day of Year (001-365)
            hour = int(t[7:9])    # UTC Hour
            minute = int(t[9:11]) # UTC Minute
            
            # Base date for the year + days passed + hours/minutes
            dt_utc = datetime(year, 1, 1) + timedelta(days=doy - 1, hours=hour, minutes=minute)
            
            # Convert to local time
            dt_local = dt_utc + timedelta(hours=tz_offset_hours)
            local_datetimes.append(dt_local)
            
    return local_datetimes

## 3. Extract AOD and Create `df_aod_long`

In [8]:
def load_sds_with_metadata(hdf_file, sds_name):
    """
    Load a Science Dataset (SDS) with its metadata including scale factor,
    fill value, and valid range.
    """
    hdf = SD(hdf_file, SDC.READ)
    sds = hdf.select(sds_name)
    
    # Get the data
    data = sds.get()
    
    # Get attributes
    attrs = sds.attributes()
    
    # Extract scaling and fill value info
    scale_factor = attrs.get('scale_factor', 1.0)
    add_offset = attrs.get('add_offset', 0.0)
    fill_value = attrs.get('_FillValue', None)
    valid_range = attrs.get('valid_range', None)
    units = attrs.get('units', 'N/A')
    long_name = attrs.get('long_name', sds_name)
    
    sds.endaccess()
    hdf.end()
    
    # Apply scale and offset
    if fill_value is not None:
        data = np.where(data == fill_value, np.nan, data)
    
    if valid_range is not None:
        lo, hi = valid_range[0], valid_range[1]
        # Apply valid range BEFORE scaling
        data = np.where((data < lo) | (data > hi), np.nan, data)

    data = data * scale_factor + add_offset
    
    metadata = {
        'scale_factor': scale_factor,
        'add_offset': add_offset,
        'fill_value': fill_value,
        'valid_range': valid_range,
        'units': units,
        'long_name': long_name
    }
    
    return data, metadata

In [9]:
from rasterio.transform import rowcol


def get_nearest_valid(data_array, orbit_idx, row, col, max_radius=5):
    """
    Return the nearest valid (non-NaN) pixel value using Chebyshev distance rings.

    Priority:
      1. Exact pixel (ring 0) — returned immediately if not NaN.
      2. Ring 1, ring 2, ... up to max_radius.
         At each ring ALL valid pixels are collected and averaged.
         This avoids arbitrary single-pixel selection when multiple
         equidistant neighbours exist at the same ring.

    Chebyshev distance: max(|dr|, |dc|)
      Ring 1 → up to 8 pixels, ring 2 → up to 16 pixels, etc.
      At MCD19A2 1 km resolution, ring k covers up to k km from the station.

    Args:
        data_array  : 2-D (h, w) or 3-D (orbits, h, w) numpy array (already scaled).
        orbit_idx   : Orbit layer index (ignored for 2-D arrays).
        row, col    : Pixel coordinates of the target station.
        max_radius  : Max Chebyshev radius in pixels / km (default 5).

    Returns:
        (value, ring)
            value  – mean of all valid pixels at the nearest non-empty ring,
                     or np.nan if nothing valid was found within max_radius.
            ring   – 0 if exact pixel used; ring number if fill was applied;
                     None if no valid pixel found.
    """
    h = data_array.shape[-2]
    w = data_array.shape[-1]

    def _get(r, c):
        return data_array[orbit_idx, r, c] if data_array.ndim == 3 else data_array[r, c]

    # --- Ring 0: exact pixel ---
    val = _get(row, col)
    if not np.isnan(val):
        return val, 0

    # --- Rings 1 .. max_radius ---
    for ring in range(1, max_radius + 1):
        valid_vals = []

        # Top and bottom rows of the Chebyshev ring square
        for dc in range(-ring, ring + 1):
            for dr in (-ring, ring):
                nr, nc = row + dr, col + dc
                if 0 <= nr < h and 0 <= nc < w:
                    v = _get(nr, nc)
                    if not np.isnan(v):
                        valid_vals.append(v)

        # Left and right columns (corners already covered above)
        for dr in range(-ring + 1, ring):
            for dc in (-ring, ring):
                nr, nc = row + dr, col + dc
                if 0 <= nr < h and 0 <= nc < w:
                    v = _get(nr, nc)
                    if not np.isnan(v):
                        valid_vals.append(v)

        if valid_vals:
            # Average ALL valid pixels in this ring (not just the first one found)
            return np.mean(valid_vals), ring

    return np.nan, None                 # entire neighbourhood is NaN


def extract_modis_for_stations(hdf_path, df_stations, target_sds=None, max_fill_radius=5):
    """
    Extract MODIS AOD values for each station from a single HDF file.

    For each variable the exact 1 km pixel overlapping the station is tried
    first.  If that pixel is NaN (cloud shadow, retrieval failure, etc.) the
    function searches outward ring-by-ring (Chebyshev) up to *max_fill_radius*
    pixels / km, averaging ALL valid pixels found at the nearest non-empty ring.
    Rows are appended only when at least one primary AOD band
    (Optical_Depth_047 or Optical_Depth_055) is non-NaN after the fill.

    Args:
        hdf_path        : Path to the MCD19A2 HDF4 file.
        df_stations     : DataFrame with columns modis_x, modis_y, ID.
        target_sds      : List of SDS layer names to extract.
        max_fill_radius : Chebyshev search radius in pixels / km (default 5).
    """
    if target_sds is None:
        target_sds = [
            'Optical_Depth_047', 'Optical_Depth_055', 'AOD_Uncertainty',
            'Column_WV', 'AngstromExp_470-780', 'AOD_QA', 'FineModeFraction',
            'Injection_Height', 'cosSZA', 'cosVZA', 'RelAZ',
            'Scattering_Angle', 'Glint_Angle'
        ]

    # Read orbit time attributes
    hdf = SD(hdf_path, SDC.READ)
    orbit_time_str = hdf.attributes().get('Orbit_time_stamp', '')
    hdf.end()

    local_times = parse_modis_orbit_times(orbit_time_str)

    # Load all requested data cubes into memory
    data_cubes = {}
    for sds_name in target_sds:
        try:
            data_array, _ = load_sds_with_metadata(hdf_path, sds_name)
            data_cubes[sds_name] = data_array
        except Exception as e:
            print(f"  [!] Warning: Could not load {sds_name}: {e}")

    if not data_cubes:
        return pd.DataFrame()

    # Use Rasterio only for the spatial index mapping
    first_sds = next(iter(data_cubes))
    sample_path = f'HDF4_EOS:EOS_GRID:"{hdf_path}":grid1km:{first_sds}'
    records = []

    with rasterio.open(sample_path) as src:
        all_rows, all_cols = rowcol(
            src.transform,
            df_stations['modis_x'].values,
            df_stations['modis_y'].values
        )
        all_rows = [int(r) for r in all_rows]
        all_cols = [int(c) for c in all_cols]

        for orbit_idx in range(len(local_times)):
            layer_time = local_times[orbit_idx]

            for i, station in enumerate(df_stations.itertuples()):
                try:
                    row, col = all_rows[i], all_cols[i]
                except IndexError:
                    print(f"  [!] Warning: Index out of bounds for station {station.ID}")
                    continue

                if not (0 <= row < src.height and 0 <= col < src.width):
                    continue

                record = {'ID': str(station.ID), 'timestamp': layer_time}
                is_valid_row = False

                for sds_name, data_array in data_cubes.items():
                    arr_h = data_array.shape[-2]
                    arr_w = data_array.shape[-1]

                    # Scale pixel coordinates if SDS resolution differs from 1 km grid
                    t_row = int(row * (arr_h / src.height))
                    t_col = int(col * (arr_w / src.width))

                    # Exact pixel first; fall back to nearest-ring average if NaN
                    val, fill_ring = get_nearest_valid(
                        data_array, orbit_idx, t_row, t_col,
                        max_radius=max_fill_radius
                    )

                    record[sds_name] = val
                    if fill_ring is not None and fill_ring > 0:
                        # Track fill ring for QA (ring ≈ km at 1 km resolution)
                        record[f'{sds_name}_fill_ring'] = fill_ring

                    if sds_name in ['Optical_Depth_047', 'Optical_Depth_055'] and not np.isnan(val):
                        is_valid_row = True

                if is_valid_row:
                    records.append(record)

    return pd.DataFrame(records)


In [10]:
hdf_directory = '/home/slow_data/Air_Quality/MODIS_MCD19A2' 
output_directory = '/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2' 

In [11]:
import glob

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# 3. Find all HDF files
hdf_files = glob.glob(os.path.join(hdf_directory, '*', '*', 'MCD19A2*.hdf'))
hdf_files.sort()

print(f"Found {len(hdf_files)} HDF files to process.")

Found 6432 HDF files to process.


In [12]:
all_dataframes = []

# 4. Loop through every HDF file and extract
for i, hdf_file in enumerate(hdf_files):
    print(f"Processing [{i+1}/{len(hdf_files)}]: {os.path.basename(hdf_file)}...")
    try:
        df_daily = extract_modis_for_stations(hdf_file, station_df)
        
        # Only append if we actually found valid data in this file
        if not df_daily.empty:
            all_dataframes.append(df_daily)
    except Exception as e:
        print(f"  [!] Error processing {os.path.basename(hdf_file)}: {e}")
        
# 5. Combine everything into one giant master DataFrame
if len(all_dataframes) > 0:
    print("\nCombining all days into master dataset...")
    df_master = pd.concat(all_dataframes, ignore_index=True)
    
    # Sort master dataset chronologically (Crucial for merge_asof)
    df_master = df_master.sort_values(by='timestamp').reset_index(drop=True)
    
    # ---Add the specific formatted string Timestamp column ---
    df_master['Timestamp_MODIS_str'] = df_master['timestamp'].dt.strftime('%Y%m%d_%H%M')
    
    # Reorder columns to make it cleaner
    cols = df_master.columns.tolist()
    cols.insert(2, cols.pop(cols.index('Timestamp_MODIS_str')))
    df_master = df_master[cols]
    
    print("\nSuccess!")
else:
    print("\n[!] No valid data was extracted from any of the HDF files.")

Processing [1/6432]: MCD19A2.A2022244.h27v06.061.2023010205026.hdf...
Processing [2/6432]: MCD19A2.A2022244.h27v07.061.2023010034912.hdf...
Processing [3/6432]: MCD19A2.A2022244.h28v06.061.2023010085933.hdf...
Processing [4/6432]: MCD19A2.A2022244.h28v07.061.2023010010321.hdf...
Processing [5/6432]: MCD19A2.A2022244.h28v08.061.2023009171129.hdf...
Processing [6/6432]: MCD19A2.A2022245.h27v06.061.2023010211621.hdf...
Processing [7/6432]: MCD19A2.A2022245.h27v07.061.2023010041629.hdf...
Processing [8/6432]: MCD19A2.A2022245.h28v06.061.2023010092006.hdf...
Processing [9/6432]: MCD19A2.A2022245.h28v07.061.2023010012335.hdf...
Processing [10/6432]: MCD19A2.A2022245.h28v08.061.2023009180603.hdf...
Processing [11/6432]: MCD19A2.A2022246.h27v06.061.2023010213050.hdf...
Processing [12/6432]: MCD19A2.A2022246.h27v07.061.2023010043808.hdf...
Processing [13/6432]: MCD19A2.A2022246.h28v06.061.2023010093550.hdf...
Processing [14/6432]: MCD19A2.A2022246.h28v07.061.2023010014510.hdf...
Processing [15/

## 4. The Temporal Merge
We use `pd.merge_asof` to map the exact MODIS timestamp (e.g. 10:15) to the nearest AQI timestamp (e.g. 10:00). We set a `tolerance` of 1 hour so it won't mistakenly map to an AQI reading from hours ago if the station went offline.

In [13]:
# df_master = df_master.sort_values(by='timestamp').reset_index(drop=True)


In [16]:
#------------------------------------------------------------------
# 6. Load and Merge AQI Data

print("Merging MODIS and AQI data (1-hour tolerance)...")
df_merged = pd.merge_asof(
    df_master,                      # Left side: MODIS data
    df_aqi,                         # Right side: Ground station data
    left_on='timestamp',            # MODIS exact time 
    right_on='Timestamp_DT',        # AQI hourly time
    by='ID',                        # Must match the same station
    direction='nearest',            # Match to nearest hour
    tolerance=pd.Timedelta('1h'),    # Maximum 1 hour gap
    
)

# Drop rows where no AQI data was found within the 1-hour window
# df_merged = df_merged.dropna(subset=['AQI'])
print(f"Data merged successfully. Total paired records: {len(df_merged)}")



Merging MODIS and AQI data (1-hour tolerance)...
Data merged successfully. Total paired records: 8075


## 5. Export CSV

In [17]:
#------------------------------------------------------------------------------------
# 7. Group by Station ID and Export separate CSVs
print(f"\nExporting time-series CSVs for each station to: {output_directory}")

grouped = df_merged.groupby('ID')

for station_id, group_df in grouped:
    # Sort the group cleanly by timestamp before saving
    group_clean = group_df.sort_values('timestamp')
    
    # Clean up the ID for safe filename generation
    safe_id = str(station_id).replace("/", "_").replace("\\", "_")
    out_path = os.path.join(output_directory, f"MODIS_AQI_{safe_id}.csv")
    
    # Save the station's dedicated time-series
    group_clean.to_csv(out_path, index=False)
    print(f"  -> Saved {out_path} ({len(group_clean)} records)")
    
print("\nSuccess! All combined station time-series files have been generated.")


Exporting time-series CSVs for each station to: /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_28505268571336961948594948504.csv (566 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_28560877461938780203765592307.csv (490 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_29195707587706641566224751462.csv (541 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_29213751141295132066317063859.csv (789 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_31387251434693138681789561386.csv (595 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_31388839920718814259329251882.csv (709 records)
  -> Saved /home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2/MODIS_AQI_31388851800421997746903202346.csv (874 records)
  ->

In [ ]:
import pandas as pd
from pathlib import Path
import re  # Fix 1: Added missing import

def extract_station_id(path: Path) -> str:
    """Pull the numeric station ID from either filename convention."""
    return re.sub(r'^MODIS_AQI_', '', path.stem)

MODIS_DIR = Path('/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v2')
METADATA_PATH = '/home/slow_data/Air_Quality/Envisoft_station_metadata.csv'

# Fix 2: Load metadata ONCE outside the loop for speed
meta = pd.read_csv(METADATA_PATH)
# Ensure ID is string to match the filename extraction
mapping = meta.set_index(meta['ID'].astype(str))['Name'].to_dict()

# Get the list of files
modis_files = {extract_station_id(f): f for f in MODIS_DIR.glob('MODIS_AQI_*.csv')}

for station_id, file_path in modis_files.items(): # Fix 3: Iterate through items (key and path)
    # Load the specific station file
    df = pd.read_csv(file_path)
    
    # Optional: Ensure the ID column in the dataframe is also a string to match the mapping
    if 'ID' in df.columns:
        df['ID'] = df['ID'].astype(str)
        
        # Fill missing 'Name' values
        df['Name'] = df['Name'].fillna(df['ID'].map(mapping))
        
        # Save back to the original file path
        df.to_csv(file_path, index=False)
        print(f"Processed Station ID: {station_id}")
    else:
        print(f"Warning: No 'ID' column found in {file_path.name}")

print("Done! All missing names have been filled.")

Processed Station ID: 31388851800421997746903202346
Processed Station ID: 31390912357075263208060500522
Processed Station ID: 31388883344354363840031242796
Processed Station ID: 28560877461938780203765592307
Processed Station ID: 31390908889087377344742439468
Processed Station ID: 31390932574706768021562473002
Processed Station ID: 29195707587706641566224751462
Processed Station ID: 31390903576425084107499649578
Processed Station ID: 29213751141295132066317063859
Processed Station ID: 31390916083317566102523755051
Processed Station ID: 31387251434693138681789561386
Processed Station ID: 31388839920718814259329251882
Processed Station ID: 28505268571336961948594948504
Done! All missing names have been filled.
